In [29]:
import sys


sys.path.append('../')

from bunkatopics import Bunka
from langchain_community.embeddings import HuggingFaceEmbeddings
from datasets import load_dataset
import random

# import umap
from umap.umap_ import UMAP # My personal Umap bugs so I use this one
from sentence_transformers import SentenceTransformer
random.seed(42)


model_name = "all-MiniLM-L6-v2"
embedding_model = HuggingFaceEmbeddings(model_name=model_name) # We recommend starting with a small model




In [30]:

#Scientific Litterature Data
dataset = load_dataset("CShorten/ML-ArXiv-Papers")["train"]["title"]
raw_docs = random.sample(dataset, 3000)


projection_model = UMAP(
                n_components=2,
                random_state=42,
                n_neighbors=5,# I want to optimise the local structure (5 low, 25 high)
                min_dist = 0.3,
                metric = 'cosine') # I don't want to disperse embeddings

# #embedding_model = SentenceTransformer(model_name_or_path="all-MiniLM-L6-v2")
# embedding_model = SentenceTransformer(model_name_or_path="Bunka/sentence_transformer_encoder")

projection_model.n_components

2

In [31]:
bunka = Bunka(embedding_model=embedding_model, 
                projection_model=projection_model)  # the language is automatically detected, make sure the embedding model is adapted

In [32]:
bunka.actual_dimensions

2

In [33]:
# Fit Bunka to your text data
bunka.fit(raw_docs)

2025-05-08 15:37:09 - Bunka - INFO - Processing 43115 tokens
2025-05-08 15:37:09 - Bunka - INFO - Detected language: English
2025-05-08 15:37:09 - Bunka - INFO - Embedding documents... (can take varying amounts of time depending on their size)
2025-05-08 15:37:10 - Bunka - INFO - Reducing dimensions to 2 using UMAP
2025-05-08 15:37:18 - Bunka - INFO - Extracting meaningful terms from documents...
2025-05-08 15:37:18 - Bunka - INFO - Sampling 2000 documents for term extraction
100%|██████████| 2000/2000 [00:09<00:00, 221.22it/s]


In [34]:
bunka.fig_embeddings

In [35]:
len(bunka.docs[0].embedding)
len(bunka.docs[0].nd_embedding)

0

In [36]:
from sklearn.cluster import HDBSCAN

max_cluster_size = int(0.02*len(raw_docs))
min_cluster_size = max(2, int(0.003*len(raw_docs)))
# min_cluster_size = 15

clustering_model = HDBSCAN(min_samples = 1, 
                max_cluster_size=max_cluster_size, 
                min_cluster_size=min_cluster_size, 
                metric = 'euclidean',
                cluster_selection_method = 'leaf')


df_topics = bunka.get_topics(n_clusters=10, name_length=5, min_count_terms = 2,  custom_clustering_model=clustering_model) # Specify the number of terms to describe each topic

2025-05-08 15:37:27 - Bunka - INFO - Computing the topics


In [37]:
bunka.visualize_topics()

2025-05-08 15:37:28 - Bunka - INFO - Creating the Bunka Map
